# §SA — Stellar Age Anisotropy · Ah Framework
**Target:** p=0.00019 (residuals), z=3.32σ
**Data:** SDSS DR17 stellarMassFSPSGranWideDust
**Upload your CSV first:** sdss_stellar_age_v4.csv

In [ ]:
import pandas as pd, numpy as np, os, sys
from scipy import stats
from astropy.coordinates import SkyCoord
import astropy.units as u
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, '../src')

AGE_FILE = 'sdss_stellar_age_v4.csv'
assert os.path.exists(AGE_FILE), 'Upload sdss_stellar_age_v4.csv first'

df = pd.read_csv(AGE_FILE)
df = df.rename(columns={'ra':'RA','dec':'Dec','z':'z','t_age':'t_age','logMass':'logMstar'})
for c in ['RA','Dec','z','t_age','logMstar']: df[c] = pd.to_numeric(df[c], errors='coerce')
df['logAge'] = np.log10(df['t_age'].clip(lower=0.001))
df = df.dropna(subset=['RA','Dec','logAge']).copy()

axis = SkyCoord(l=220*u.deg, b=-20*u.deg, frame='galactic')
coords = SkyCoord(ra=df.RA.values*u.deg, dec=df.Dec.values*u.deg, frame='icrs')
df['sep'] = coords.separation(axis).deg
df_z = df[(df.z>0.05)&(df.z<0.20)].copy()

# Method 1
groove = df_z[(df_z.sep>=30)&(df_z.sep<65)]['logAge'].values
equat  = df_z[(df_z.sep>=65)&(df_z.sep<120)]['logAge'].values
_, p1  = stats.mannwhitneyu(groove, equat, alternative='greater')

# Method 2 residuals
X = np.column_stack([df_z['logMstar'].fillna(df_z['logMstar'].median()), df_z['z']])
y = df_z['logAge'].values
A = np.column_stack([np.ones(len(X)), X])
c,_,_,_ = np.linalg.lstsq(A, y, rcond=None)
df_z = df_z.copy(); df_z['resid'] = y - A@c
g_r = df_z[(df_z.sep>=30)&(df_z.sep<65)]['resid'].values
e_r = df_z[(df_z.sep>=65)&(df_z.sep<120)]['resid'].values
_, p2 = stats.mannwhitneyu(g_r, e_r, alternative='greater')
rng=np.random.default_rng(99); r_all=df_z['resid'].values; s_all=df_z.sep.values
gm=(s_all>=30)&(s_all<65); em=(s_all>=65)&(s_all<120)
obs_d=np.mean(r_all[gm])-np.mean(r_all[em])
perm=np.array([np.mean((a:=rng.permutation(r_all))[gm])-np.mean(a[em]) for _ in range(5000)])
z_mc=(obs_d-perm.mean())/perm.std()

# Method 3 matched
g_df=df_z[(df_z.sep>=30)&(df_z.sep<65)].dropna(subset=['logMstar','z','logAge'])
e_df=df_z[(df_z.sep>=65)&(df_z.sep<120)].dropna(subset=['logMstar','z','logAge'])
sc=StandardScaler(); e_sc=sc.fit_transform(e_df[['logMstar','z']]); g_sc=sc.transform(g_df[['logMstar','z']])
nn=NearestNeighbors(n_neighbors=1); nn.fit(e_sc); dist,idx=nn.kneighbors(g_sc)
good=dist.flatten()<0.5
delta_m=g_df['logAge'].values[good]-e_df['logAge'].values[idx.flatten()[good]]
_,p3w=stats.wilcoxon(delta_m, alternative='greater')

print(f'Method 1 Mann-Whitney:  p={p1:.5f}  target=0.026')
print(f'Method 2 residuals:     p={p2:.5f}  target=0.00019  z={z_mc:.2f}σ target=3.32σ')
print(f'Method 3 Wilcoxon:      p={p3w:.5f}  target=0.00085')
print()
print('='*45)
if p1<0.05 and p2<0.05 and p3w<0.05:
    print('ALL METHODS CONFIRMED ✓')
else:
    print('CHECK VALUES')
print('='*45)